# 02 · How many synergies? (Q1)

**Owner:** Person B · **Read first:** Lecture 6 (PCA), Lecture 3 p. 23–27 (NMF), reading 0922 (Cunningham & Yu)

**Question:** how many synergies does it take to explain the EMG?
**We expect:** 3–5 synergies explain at least 90% of the EMG (VAF ≥ 90%). **Wrong if** we need close to 16.

**Model:** `EMG ≈ c(t) × W`. In scikit-learn: `W = nmf.components_` (k × 16), `c = nmf.transform(EMG)` (T × k).

In [1]:
# ── Setup: works on Colab and locally ─────────────────────────────────────────
# After this cell the working directory is the repo root, wherever you run from.
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ or Path("/content").exists()
if IN_COLAB:
    ROOT = Path("/content/emg2pose")
    if not ROOT.exists():
        !git clone https://github.com/Wilbertbh-Tan/emg2pose.git {ROOT}
    !pip install -q h5py
else:
    ROOT = Path.cwd().resolve()
    while not (ROOT / "setup.py").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    assert (ROOT / "setup.py").exists(), "Run this notebook from inside the emg2pose repo"
os.chdir(ROOT)

PROCESSED = ROOT / "data" / "processed_mini.npz"   # made by notebook 01 (data/ is never committed)

In [2]:
# ── Load the shared data made by notebook 01 (runs 01 first if needed) ───────
import numpy as np
if not PROCESSED.exists():
    %run notebooks/01_data_prep/01_data_prep.ipynb
d = np.load(PROCESSED)

emg        = d["emg"]          # (N, 16)  EMG envelope, 50 samples/s, non-negative
pose       = d["pose"]         # (N, 20)  joint angles in degrees
pose_valid = d["pose_valid"]   # (N,)     False where pose tracking failed
rec        = d["rec"]          # (N,)     which recording (0–29) each row comes from
split      = d["split"]        # (N,)     0 = train, 1 = test, -1 = gap (use neither)
stage_of_rec, side_of_rec = d["stage"], d["side"]   # (30,) task name and hand per recording

train, test = split == 0, split == 1
SEED = 0   # pass random_state=SEED to every model so results repeat
print(f"{len(emg)} frames | {train.sum()} train | {test.sum()} test | {len(stage_of_rec)} recordings")

105484 frames | 84376 train | 19608 test | 30 recordings


### The variables you start with
`N` = number of frames in all 30 recordings together (≈ 90,000: 30 recordings × ~60 s × 50 frames/s).
Every row of every array is one frame, 20 ms long. Boolean arrays are **masks**: `emg[train]` keeps only the train rows.

| Name | Shape | dtype | What it is |
|---|---|---|---|
| `emg` | (N, 16) | float32 | EMG envelope, one column per electrode, ≥ 0, roughly 0–1 |
| `pose` | (N, 20) | float32 | joint angles in **degrees**, one column per joint |
| `pose_valid` | (N,) | bool | False where pose tracking failed (don't use those `pose` rows) |
| `rec` | (N,) | int16 | which recording (0–29) the frame comes from |
| `split` | (N,) | int8 | 0 = train, 1 = test, −1 = gap between them (use neither) |
| `train`, `test` | (N,) | bool | `split == 0`, `split == 1` |
| `stage_of_rec` | (30,) | str | task name of each recording; per frame: `stage_of_rec[rec]` → (N,) |
| `side_of_rec` | (30,) | str | `"left"` or `"right"` per recording; per frame: `side_of_rec[rec]` → (N,) |

Letters used in the docstrings below: **T** = number of frames in the array you pass in (any subset of N),
**k** = number of synergies, **C** = number of gesture classes. Run a helper on a tiny made-up array first
(e.g. `np.random.rand(5, 16)`) to check it returns the shape its docstring promises.

## Before you run anything: your prediction

Write what you expect and why (2–3 sentences). Compare at the end. Surprises are the interesting part.

> *Prediction:* I think the structure should be confined to some latent states that are reflective othe muscles. I think geometric should reflect better and if there is some way to include some time variance that would be work even better. 

## Task 1 · A VAF function
VAF = 1 − (squared reconstruction error) / (squared data), as a percentage. It is the standard
"how much of the EMG is explained" number in the synergy literature.

In [ ]:
import numpy as np

def vaf(X: np.ndarray, X_hat: np.ndarray) -> float:
    """Variance accounted for, in %.

    Parameters
    ----------
    X     : (T, 16) float, the real EMG (non-negative)
    X_hat : (T, 16) float, the reconstruction of X from the synergies (same shape as X)

    Returns
    -------
    float between −∞ and 100. 100 = perfect reconstruction.
    Uncentered: compare the error against X itself, not X minus its mean.

    Check: vaf(X, X) → 100.0 and vaf(X, np.zeros_like(X)) → 0.0
    """
    # TODO
    raise NotImplementedError


def reconstruct(model, X: np.ndarray) -> np.ndarray:
    """Rebuild X from a fitted NMF or PCA model.

    Parameters
    ----------
    model : a fitted sklearn NMF or PCA with k components
    X     : (T, 16) float, EMG to encode and then rebuild

    Returns
    -------
    X_hat : (T, 16) float
        NMF: c(t) = model.transform(X) is (T, k), W = model.components_ is (k, 16); combine them.
        PCA: model.inverse_transform does the rebuilding for you.
    """
    # TODO
    raise NotImplementedError

## Task 2 · NMF and PCA for k = 1 … 10, scored on test data
Fit on `emg[train]`, then reconstruct `emg[test]` and compute VAF. Use several random starts (`random_state`)
for NMF, because NMF can land on different answers.

In [ ]:
from sklearn.decomposition import NMF, PCA

ks = list(range(1, 11))
vaf_nmf: list[float] = []      # length 10 when done: test VAF (%) for k = 1 … 10
vaf_pca: list[float] = []
nmf_models: dict = {}          # k → fitted NMF   (Task 6 and 7 reuse them)
pca_models: dict = {}          # k → fitted PCA

X_train = emg[train]           # (T_train, 16)
X_test  = emg[test]            # (T_test, 16)

for k in ks:
    # TODO: fit NMF(n_components=k, random_state=SEED, max_iter=1000) on X_train; store it in nmf_models[k]
    # TODO: append vaf(X_test, reconstruct(...)) to vaf_nmf
    # TODO: the same with PCA into pca_models and vaf_pca
    ...

## Task 3 · Plot VAF vs k and choose k

In [ ]:
# TODO: one figure: VAF (%) against k (x = ks, y = vaf_nmf and vaf_pca), and a dashed line at 90%
# TODO: choose k where the NMF curve flattens; write it below

K_CHOSEN: int | None = None    # e.g. 4

## Task 4 · Look at the synergies
For the chosen k: a bar chart per synergy of its 16 channel weights (normalise each synergy so its
weights sum to 1, so they are comparable). Also plot c(t) for one recording next to a few joint angles:
when does each synergy switch on?

In [ ]:
# W = nmf_models[K_CHOSEN].components_      → (k, 16): row i = synergy i, one weight per electrode
# c = nmf_models[K_CHOSEN].transform(emg)   → (N, k):  column i = how active synergy i is at each frame
r = 0                                         # the recording to plot
in_r = rec == r                               # (N,) bool mask for that recording

# TODO: bar chart per synergy of its 16 weights (divide each row of W by its sum first)
# TODO: c[in_r] (T_r, k) over time, with 3 columns of pose[in_r] (T_r, 20) underneath on the same time axis

## Result
- Chosen k:
- VAF at that k (NMF / PCA):
- Was the prediction right? Why or why not?